In [1]:
import json
from pathlib import Path
import torch
from sentence_transformers import SentenceTransformer
import numpy as np

/home/gusevsaint/Workspace/study/practice/mental-helper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
data_path = Path("../data/qa_dataset.json")
with open(data_path, "r", encoding="utf-8") as f:
    qa_data = json.load(f)

In [3]:
# Load E5-small-v2
model_name = "intfloat/e5-small-v2"
model = SentenceTransformer(model_name)

In [ ]:
# Extract all questions and generate embeddings
# E5 model requires "passage:" prefix for documents being indexed

questions = [entry["question"] for entry in qa_data]
embeddings = model.encode(
    [f"passage: {q}" for q in questions],
    convert_to_tensor=True,
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True,
)

Batches: 100%|██████████| 220/220 [00:42<00:00,  5.21it/s]


In [ ]:
# Save embeddings
index_path = Path("../data/e5_index.pt")
cache = {
    "model_name": model_name,
    "embeddings": embeddings.cpu(),  # Move to CPU for storage
}
torch.save(cache, index_path)